# Lab 2: Routing & Resilience

In this lab, you will build an agent system that is both **efficient** and **resilient**. You will solve the problem of "tool overload" using semantic routing and the problem of "infinite loops" using observability and circuit breakers.

## 🎯 Learning Objectives
1. **Dynamic Tool Routing:** Select only the most relevant tools for a query to keep context windows lean.
2. **Observability (Tracing):** Instrument your agent to capture a detailed trace of reasoning and actions.
3. **Resilience (Loop Detection):** Implement a "circuit breaker" that detects repetitive behaviors and stops loops before they burn tokens.

---

## 🏗️ The Architecture
We are building a **Routed Agent** that follows this flow:
1. **Perception:** User query enters.
2. **Routing:** 
   - A cheap classifier routes to a domain (financial, academic, general).
   - An embedding-based selector finds the top-K most relevant tools.
3. **Execution:** The agent loop runs with ONLY the selected tools.
4. **Monitoring:** A Tracer logs every step.
5. **Circuit Breaker:** A Loop Detector checks for stagnation or repetition.

---

## Phase 0: Setup and Environment

Run these cells to load configs and mock tools.

In [ ]:
import json
import time
import logging
from typing import List, Dict, Any, Callable
from pydantic import BaseModel, create_model
from litellm import completion
import inspect
import numpy as np

import nest_asyncio
nest_asyncio.apply()

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Central model configuration
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o")
EMBEDDING_MODEL = "text-embedding-3-small"

import dotenv
dotenv.load_dotenv()


In [ ]:
import numpy as np

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Calculate the cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    if np.linalg.norm(a) == 0 or np.linalg.norm(b) == 0:
        return 0.0
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def jaccard_similarity(s1: str, s2: str) -> float:
    """Calculate the Jaccard similarity between two strings."""
    set1 = set(s1.lower().split())
    set2 = set(s2.lower().split())
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)


In [ ]:
from typing import Callable, Any, Dict
from pydantic import BaseModel, create_model
import inspect

class Tool:
    """A callable tool with schema."""
    def __init__(self, name: str, func: Callable, description: str):
        self.name = name
        self.func = func
        self.description = description
        self.model = self._create_pydantic_model(func)

    def _create_pydantic_model(self, func: Callable) -> type[BaseModel]:
        sig = inspect.signature(func)
        fields = {}
        for name, param in sig.parameters.items():
            if name == "self": continue
            annotation = param.annotation if param.annotation != inspect.Parameter.empty else str
            fields[name] = (annotation, ... if param.default == inspect.Parameter.empty else param.default)
        return create_model(f"{self.name}Schema", **fields)

    def to_openai_schema(self) -> dict:
        schema = self.model.model_json_schema()
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": schema.get("properties", {}),
                    "required": schema.get("required", []),
                    "additionalProperties": False,
                },
                "strict": True
            },
        }

    def execute(self, **kwargs) -> Any:
        validated_args = self.model(**kwargs)
        return self.func(**validated_args.model_dump())

class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}
        self._categories: Dict[str, list[str]] = {}

    def register(self, name: str, description: str, category: str = "general"):
        def decorator(func: Callable):
            tool = Tool(name, func, description)
            self._tools[name] = tool
            if category not in self._categories: self._categories[category] = []
            self._categories[category].append(name)
            return func
        return decorator

    def get_tool(self, name: str) -> Tool | None: return self._tools.get(name)
    def get_all_tools(self) -> list[Tool]: return list(self._tools.values())
    def get_tools_by_category(self, category: str) -> list[Tool]:
        return [self._tools[name] for name in self._categories.get(category, [])]

registry = ToolRegistry()


logger = logging.getLogger(__name__)

def search(query: str) -> str:
    """A search tool that returns errors for certain queries (simulates failure)."""
    mock_results = {
        "capital of france": "Paris is the capital of France.",
        "population of paris": "The population of Paris is approximately 2.1 million.",
        "python programming": "Python is a high-level programming language.",
    }
    query_lower = query.lower()
    for key, value in mock_results.items():
        if key in query_lower:
            return value

    # For unknown queries, returns an error that can trick an agent into looping
    return f"Error: No results found for '{query}'. Try searching with different keywords."


def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        allowed = set('0123456789+-*/.(). ')
        if all(c in allowed for c in expression):
            return str(eval(expression))
        return "Error: Invalid expression"
    except Exception as e:
        return f"Error: {e}"

TOOLS_DICT = {"search": search, "calculate": calculate}

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search for information. Returns text results.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression"}
                },
                "required": ["expression"],
            },
        },
    },
]


## Phase 1: Semantic Tool Selection
Implement the embedding-based selection logic.

In [ ]:
"""
Lab 2 - Step 1: Semantic Tool Selector
======================================
"""

import numpy as np
from litellm import embedding
from tools.registry import registry, Tool
from config import EMBEDDING_MODEL
from utils.math_utils import cosine_similarity

def get_embedding_vector(text: str) -> list[float]:
    """Get embedding vector using LiteLLM."""
    response = embedding(model=EMBEDDING_MODEL, input=[text])
    return response.data[0]["embedding"]

class SemanticToolSelector:
    def __init__(self):
        self._tool_embeddings: dict[str, list[float]] = {}
        self._indexed = False

    def build_index(self):
        """TODO: Embed all registered tool descriptions."""
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---

    def select_tools(self, query: str, top_k: int = 5) -> list[tuple[Tool, float]]:
        """TODO: Select the top-K most relevant tools for a query."""
        # --- YOUR CODE HERE ---
        return []
        # --- END YOUR CODE ---

    def get_tool_schemas(self, query: str, top_k: int = 5) -> list[dict]:
        selected = self.select_tools(query, top_k)
        return [tool.to_openai_schema() for tool, _score in selected]


<details>
<summary><b>Click here to see the solution</b></summary>

```python

```
</details>

## Phase 2: Agent Tracing
Implement AgentTracer to capture logic.

In [ ]:
"""
Lab 2 - Step 2: Agent Tracer
==============================
"""

import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class ToolCallRecord:
    tool_name: str
    tool_input: dict
    tool_output: str
    duration_ms: float

@dataclass
class AgentStep:
    step_number: int
    reasoning: Optional[str]
    tool_calls: list[ToolCallRecord] = field(default_factory=list)
    duration_ms: float = 0.0
    timestamp: float = field(default_factory=time.time)

@dataclass
class Trace:
    trace_id: str
    agent_name: str
    input_query: str
    steps: list[AgentStep] = field(default_factory=list)
    status: str = "running"

class AgentTracer:
    def __init__(self, verbose: bool = True):
        self._traces: dict[str, Trace] = {}
        self.verbose = verbose

    def start_trace(self, agent_name: str, query: str) -> str:
        """TODO: Start a new trace."""
        # --- YOUR CODE HERE ---
        return ""
        # --- END YOUR CODE ---

    def log_step(self, trace_id: str, step: AgentStep):
        """TODO: Log a completed step."""
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---

    def end_trace(self, trace_id: str, status: str = "completed"):
        """TODO: Finalize the trace."""
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---

    def print_summary(self, trace_id: str):
        """TODO: Print a human-readable summary."""
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---


<details>
<summary><b>Click here to see the solution</b></summary>

```python
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class ToolCallRecord:
    tool_name: str
    tool_input: dict
    tool_output: str
    duration_ms: float

@dataclass
class AgentStep:
    step_number: int
    reasoning: Optional[str]
    tool_calls: list[ToolCallRecord] = field(default_factory=list)
    duration_ms: float = 0.0
    timestamp: float = field(default_factory=time.time)

@dataclass
class Trace:
    trace_id: str
    agent_name: str
    input_query: str
    steps: list[AgentStep] = field(default_factory=list)
    status: str = "running"

class AgentTracer:
    def __init__(self, verbose: bool = True):
        self._traces: dict[str, Trace] = {}
        self.verbose = verbose

    def start_trace(self, agent_name: str, query: str) -> str:
        trace_id = str(uuid.uuid4())[:8]
        self._traces[trace_id] = Trace(trace_id, agent_name, query)
        if self.verbose: print(f"[Trace {trace_id}] Started: {agent_name}")
        return trace_id

    def log_step(self, trace_id: str, step: AgentStep):
        if trace_id in self._traces:
            self._traces[trace_id].steps.append(step)
            if self.verbose: print(f"  Step {step.step_number}: {step.duration_ms:.0f}ms")

    def end_trace(self, trace_id: str, status: str = "completed"):
        if trace_id in self._traces:
            self._traces[trace_id].status = status
            if self.verbose: print(f"[Trace {trace_id}] {status.upper()}")

    def print_summary(self, trace_id: str):
        trace = self._traces.get(trace_id)
        if trace:
            print(f"\n--- Summary: {trace.agent_name} ---")
            print(f"Query: {trace.input_query}")
            print(f"Steps: {len(trace.steps)}, Status: {trace.status}")

```
</details>

## Phase 3: Loop Detection
Implement strategies to catch infinite loops.

In [ ]:
"""
Lab 2 - Step 3: Advanced Loop Detector
========================================
"""

from dataclasses import dataclass
from utils.math_utils import jaccard_similarity

@dataclass
class LoopDetectionResult:
    is_looping: bool
    strategy: str   # "exact", "fuzzy", "stagnation", "none"
    message: str

class AdvancedLoopDetector:
    def __init__(self, exact_threshold: int = 2, fuzzy_threshold: float = 0.8, stagnation_window: int = 3):
        self.exact_threshold = exact_threshold
        self.fuzzy_threshold = fuzzy_threshold
        self.stagnation_window = stagnation_window
        self.tool_history: list[tuple[str, str]] = []
        self.output_history: list[str] = []

    def _jaccard_similarity(self, s1: str, s2: str) -> float:
        """Helper to calculate word-level Jaccard similarity."""
        return jaccard_similarity(s1, s2)

    def check_tool_call(self, tool_name: str, tool_input: str) -> LoopDetectionResult:
        """TODO: Check for exact and fuzzy loops."""
        # --- YOUR CODE HERE ---
        return LoopDetectionResult(False, "none", "")
        # --- END YOUR CODE ---

    def check_output_stagnation(self, output: str) -> LoopDetectionResult:
        """TODO: Check for repetitive reasoning outputs."""
        # --- YOUR CODE HERE ---
        return LoopDetectionResult(False, "none", "")
        # --- END YOUR CODE ---


<details>
<summary><b>Click here to see the solution</b></summary>

```python
from dataclasses import dataclass
from utils.math_utils import jaccard_similarity

@dataclass
class LoopDetectionResult:
    is_looping: bool
    strategy: str
    message: str

class AdvancedLoopDetector:
    def __init__(self, exact_threshold: int = 2, fuzzy_threshold: float = 0.8, stagnation_window: int = 3):
        self.exact_threshold = exact_threshold
        self.fuzzy_threshold = fuzzy_threshold
        self.stagnation_window = stagnation_window
        self.tool_history: list[tuple[str, str]] = []
        self.output_history: list[str] = []

    def _jaccard_similarity(self, s1: str, s2: str) -> float:
        return jaccard_similarity(s1, s2)

    def check_tool_call(self, tool_name: str, tool_input: str) -> LoopDetectionResult:
        current = (tool_name, tool_input.strip())
        exact_count = sum(1 for t, i in self.tool_history if (t, i.strip()) == current)
        if exact_count >= self.exact_threshold:
            return LoopDetectionResult(True, "exact", f"Exact loop: {tool_name} repeated.")
        
        fuzzy_matches = sum(1 for t, i in self.tool_history[-5:] if t == tool_name and self._jaccard_similarity(tool_input, i) >= self.fuzzy_threshold)
        if fuzzy_matches >= self.exact_threshold:
            return LoopDetectionResult(True, "fuzzy", f"Fuzzy loop: {tool_name} with similar args.")
            
        self.tool_history.append(current)
        return LoopDetectionResult(False, "none", "")

    def check_output_stagnation(self, output: str) -> LoopDetectionResult:
        self.output_history.append(output)
        if len(self.output_history) >= self.stagnation_window:
            recent = self.output_history[-self.stagnation_window:]
            sims = [self._jaccard_similarity(recent[i], recent[j]) for i in range(len(recent)) for j in range(i+1, len(recent))]
            if sum(sims)/len(sims) >= self.fuzzy_threshold:
                return LoopDetectionResult(True, "stagnation", "Reasoning outputs are stagnating.")
        return LoopDetectionResult(False, "none", "")

```
</details>

## Phase 4: Broken Agent
Wire everything together here.

In [ ]:
"""
Lab 2 - Step 4: The Broken Agent
==================================
Your job: instrument this agent with the tracer, add loop detection, and fix it.
"""

import json
import time
import logging
from litellm import completion
from config import MODEL_NAME
from tools.mock_tools import TOOLS_DICT, TOOLS_SCHEMA
from agent.tracer import AgentTracer, AgentStep, ToolCallRecord
from agent.loop_detector import AdvancedLoopDetector
from routing.semantic_router import SemanticToolSelector

# Configure Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

def run_agent(query: str, max_steps: int = 10) -> dict:
    """Standard agent loop. Needs instrumentation."""
    messages = [
        {"role": "system", "content": "You are a research assistant. Use tools to answer questions."},
        {"role": "user", "content": query},
    ]
    
    # TODO: Initialize tracer and loop detector (Step 4)
    # --- YOUR CODE HERE ---
    
    # Optional: Use SemanticToolSelector to filter tools for efficiency (Step 1)
    # selector = SemanticToolSelector()
    # tools_schema = selector.get_tool_schemas(query, top_k=5)
    
    # --- END YOUR CODE ---

    for step in range(max_steps):
        step_start = time.time()
        
        response = completion(
            model=MODEL_NAME,
            messages=messages,
            tools=TOOLS_SCHEMA, # Or your filtered tools
            tool_choice="auto",
        )

        message = response.choices[0].message
        content = message.content
        tool_calls = message.tool_calls
        messages.append(message)

        if content: logger.info(f"[Step {step + 1}] {content[:100]}...")

        if tool_calls:
            for tc in tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                args_str = json.dumps(fn_args)

                # TODO: Check for loops BEFORE executing (Step 4)
                # --- YOUR CODE HERE ---
                
                # Execute tool
                tool_start = time.time()
                result = TOOLS_DICT.get(fn_name, lambda **_: "Unknown tool")(**fn_args)
                tool_duration = (time.time() - tool_start) * 1000
                
                logger.info(f"[Step {step + 1}] Tool: {fn_name}({args_str}) -> {result[:50]}...")

                messages.append({"tool_call_id": tc.id, "role": "tool", "name": fn_name, "content": result})
                
                # --- END YOUR CODE ---

        # TODO: Log step to tracer (Step 4)
        
        if not tool_calls and content:
            # TODO: End trace and return
            return {"answer": content, "total_steps": step + 1}

    return {"answer": "[Max steps reached]", "total_steps": max_steps}

if __name__ == "__main__":
    print("Test 1: Normal Query")
    run_agent("What is the capital of France?")
    
    print("\nTest 2: Looping Query (BUG!)")
    run_agent("What are the latest trends in quantum computing?")


<details>
<summary><b>Click here to see the solution</b></summary>

```python
import json
import time
import logging
from litellm import completion
from config import MODEL_NAME
from tools.mock_tools import TOOLS_DICT, TOOLS_SCHEMA
from tracer import AgentTracer, AgentStep, ToolCallRecord
from loop_detector import AdvancedLoopDetector
from semantic_router import SemanticToolSelector

# Configure Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

def run_agent(query: str, max_steps: int = 10) -> dict:
    messages = [
        {"role": "system", "content": "You are a research assistant. Use tools to answer questions. If a tool fails, try a different approach."},
        {"role": "user", "content": query},
    ]
    
    tracer = AgentTracer()
    trace_id = tracer.start_trace("fixed_agent", query)
    loop_detector = AdvancedLoopDetector()
    selector = SemanticToolSelector()
    
    # Dynamically select tools
    tools_schema = selector.get_tool_schemas(query, top_k=2)

    for step in range(max_steps):
        step_start = time.time()
        
        response = completion(model=MODEL_NAME, messages=messages, tools=tools_schema, tool_choice="auto")
        message = response.choices[0].message
        content = message.content
        tool_calls = message.tool_calls
        messages.append(message)

        tool_records = []
        if tool_calls:
            for tc in tool_calls:
                fn_name, fn_args = tc.function.name, json.loads(tc.function.arguments)
                
                loop_check = loop_detector.check_tool_call(fn_name, json.dumps(fn_args))
                if loop_check.is_looping:
                    result = f"CIRCUIT BREAKER: {loop_check.message}"
                    tool_records.append(ToolCallRecord(fn_name, fn_args, result, 0))
                else:
                    t_start = time.time()
                    result = TOOLS_DICT.get(fn_name, lambda **_: "Tool error")(**fn_args)
                    tool_records.append(ToolCallRecord(fn_name, fn_args, result, (time.time()-t_start)*1000))
                
                messages.append({"tool_call_id": tc.id, "role": "tool", "name": fn_name, "content": result})

        tracer.log_step(trace_id, AgentStep(step+1, content, tool_records, (time.time()-step_start)*1000))
        
        if not tool_calls and content:
            tracer.end_trace(trace_id)
            tracer.print_summary(trace_id)
            return {"answer": content, "total_steps": step + 1}

    tracer.end_trace(trace_id, "failed")
    return {"answer": "[Max steps reached]", "total_steps": max_steps}

if __name__ == "__main__":
    run_agent("What is the capital of France?")
    run_agent("What are the latest trends in quantum computing?")

```
</details>